## **import Libraries**

In [188]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, classification_report, confusion_matrix

## Load Cleaned dataset

In [167]:
df = pd.read_csv("../dataset/cleaned_dataset/cyber_attack_dataset_cleaned.csv")

## Single feature encoding manually

In [ ]:
# protocol encoding for the protocol column
df['protocol'] = df['protocol'].map({'TCP':0, 'UDP':1})

## Show TOP 5 row

In [169]:
df.head()

,duration,src_bytes,dst_bytes,packet_count,protocol,failed_logins,attack_type
0,1,8605,418,631,0,0,DDoS
1,1,499,148,131,1,0,PortScan
2,10,370,160,105,1,0,PortScan
3,2,5138,320,666,0,0,DDoS
4,36,524,467,58,1,10,BruteForce


## Separate Feature and Target

In [ ]:
x = df.drop('attack_type', axis=1)
y = df['attack_type']

In [171]:
df.columns

Index(['duration', 'src_bytes', 'dst_bytes', 'packet_count', 'protocol',
       'failed_logins', 'attack_type'],
      dtype='str')

## Target encoding

In [172]:
#Target encoding for the attack_type column
le = LabelEncoder()
y = le.fit_transform(df['attack_type'])


## train test split

In [173]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    x,y, 
    test_size=0.2,
    random_state=42
    )

## Scalling

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## **Model Selection**

In [ ]:
best_model_name = None
best_model = None
best_accuracy = 0
best_recall = 0
best_classification_report = None
best_confusion_matrix = None

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average='weighted')

    print(f"{name}: {accuracy:.4f}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model_name = name
        best_model = model
        best_recall = recall
        best_confusion_matrix = confusion_matrix(y_test, y_pred)
        best_classification_report = classification_report(y_test, y_pred)

print("\n" + "=" * 40)
print("Best Model:", best_model_name)
print("Best Accuracy:", round(best_accuracy, 4))
print("Best Recall:", round(best_recall, 4))
print("Confusion Matrix:\n", best_confusion_matrix)
print("\nClassification Report:\n", best_classification_report)
print("=" * 40)

Logistic Regression: 0.9992
Random Forest: 1.0000
Decision Tree: 1.0000
Support Vector Machine: 0.9994
K-Nearest Neighbors: 0.9998
XGBoost: 1.0000

Best Model: Random Forest
Best Accuracy: 1.0
Best Recall: 1.0
Confusion Matrix:
 [[4984    0    0    0]
 [   0 5032    0    0]
 [   0    0 4984    0]
 [   0    0    0 5000]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      4984
           1       1.00      1.00      1.00      5032
           2       1.00      1.00      1.00      4984
           3       1.00      1.00      1.00      5000

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000



## Verification

In [178]:
# to verify the model performance on training and testing data
print("Train Score:", best_model.score(X_train, y_train))
print("Test Score:", best_model.score(X_test, y_test))

Train Score: 1.0
Test Score: 1.0


In [179]:
# to verify data leakage
print(x.columns.tolist())

['duration', 'src_bytes', 'dst_bytes', 'packet_count', 'protocol', 'failed_logins']


In [ ]:
print(x.shape)
print(y.shape)

(100000, 6)
(100000,)


In [ ]:
# to check correlation of features with the target variable
df_encoded = df.copy()
df_encoded['attack_type'] = le.fit_transform(df_encoded['attack_type'])
print(df_encoded.corr()['attack_type'].sort_values(ascending=False))

attack_type      1.000000
dst_bytes        0.067639
packet_count    -0.074087
duration        -0.209518
protocol        -0.226310
src_bytes       -0.234630
failed_logins   -0.717309
Name: attack_type, dtype: float64


In [187]:
# to check the model performance using cross validation
scores = cross_val_score(
    RandomForestClassifier(),
    x,
    y,
    cv=5
)

print(scores)
print(scores.mean())

[1.      0.99995 1.      1.      1.     ]
0.99999
